In [ ]:
import time
from typing import List, Dict, Any, Optional
from elasticsearch import Elasticsearch, helpers
from FlagEmbedding import BGEM3FlagModel

class BGEElasticManager:
    def __init__(
        self, 
        model_path: str = "../model/bge-m3", 
        es_url: str = "http://localhost:9200",
        use_fp16: bool = True
    ):
        self.es = Elasticsearch(es_url)
        self.model = BGEM3FlagModel(model_name_or_path=model_path, use_fp16=use_fp16)

    def list_indices(self) -> list:
        """
        Elasticsearch에 존재하는 인덱스 목록을 반환
        
        Returns:
            List[str]: 인덱스 이름 리스트
        """
        indices = self.es.cat.indices(format="json")
        return [idx["index"] for idx in indices]

    def create_index(self, index_name: str, dim: int = 1024):
        """
        Hybrid Search용 Elasticsearch index 생성
        
        구조:
        - page_content: text + keyword
        - dense_vector: dense embedding (bge-m3)
        - sparse_vector: sparse embedding (bge-m3)
        - metadata: 필터링용 JSON
        
        Args:
            index_name (str): 생성할 인덱스 이름
            dim (int): dense vector dimension (bge-m3는 기본 1024)
        """

        if self.es.indices.exists(index=index_name):
            print(f"[INFO] Index already exists: {index_name}")
            return

        mapping = {
            "settings": {
                "number_of_shards": 1,
                "number_of_replicas": 1,
                "analysis": {
                    "analyzer": {
                        "default": {
                            "type": "standard"
                        }
                    }
                }
            },
            "mappings": {
                "dynamic_templates": [
                    {
                        # ✅ metadata 내부 모든 필드를 keyword로 강제
                        "metadata_as_keyword": {
                            "path_match": "metadata.*",
                            "mapping": {
                                "type": "keyword",
                                "ignore_above": 256
                            }
                        }
                    }
                ],
                "properties": {
                    # ✅ 본문
                    "page_content": {
                        "type": "text",
                        "fields": {
                            "keyword": {
                                "type": "keyword",
                                "ignore_above": 256
                            }
                        }
                    },

                    # ✅ Dense vector (semantic search)
                    "dense_vector": {
                        "type": "dense_vector",
                        "dims": dim,
                        "index": True,
                        "similarity": "cosine"
                    },

                    # ✅ Sparse vector (BM25 + expansion)
                    "sparse_vector": {
                        "type": "sparse_vector"
                    },

                    # ✅ Metadata (필터링용)
                    "metadata": {
                        "type": "object",
                        "dynamic": True   # 다양한 key 허용
                    }
                }
            }
        }

        self.es.indices.create(index=index_name, body=mapping)
        print(f"[SUCCESS] Index created: {index_name}")

    def _normalize_metadata(self, metadata: dict) -> dict:
        """
        - flatten
        - 모든 값을 리스트 또는 문자열로 정규화
        """

        if not metadata:
            return {}

        flat = {}

        global_meta = metadata.get("global_metadata", {})
        local_meta = {k: v for k, v in metadata.items() if k != "global_metadata"}

        merged = {}
        merged.update(global_meta)
        merged.update(local_meta)

        for k, v in merged.items():

            # ✅ 리스트 그대로 유지
            if isinstance(v, list):
                flat[k] = [str(x) for x in v]

            # ✅ 단일 값도 문자열로
            else:
                flat[k] = str(v)

        return flat

    def _generate_actions(self, index_name: str, raw_documents: List[Any]):
        """Bulk 전송을 위한 Generator 구성 (bge-m3 + hybrid 구조)"""

        contents = [doc.page_content for doc in raw_documents]

        # 1. Batch Encoding (dense + sparse)
        outputs = self.model.encode(
            contents,
            return_dense=True,
            return_sparse=True
        )

        dense_vecs = outputs.get("dense_vecs")
        lexical_weights = outputs.get("lexical_weights")

        for i, doc in enumerate(raw_documents):

            # ✅ sparse_vector 형식 맞추기
            sparse_dict = {
                str(k): float(v)
                for k, v in lexical_weights[i].items()
                if v > 0
            }

            # ✅ metadata 구조 정리 (핵심)
            metadata = {}

            if hasattr(doc, "metadata") and doc.metadata:
                # global_metadata 분리 처리
                global_meta = doc.metadata.get("global_metadata", {})
                local_meta = {
                    k: v for k, v in doc.metadata.items()
                    if k != "global_metadata"
                }

                # 병합 (global → local 우선순위)
                metadata.update(global_meta)
                metadata.update(local_meta)

            metadata = self._normalize_metadata(metadata)

            # ✅ 최종 document 구조 (mapping과 일치)
            source = {
                "page_content": doc.page_content,
                "dense_vector": dense_vecs[i].tolist(),
                "sparse_vector": sparse_dict,
                "metadata": metadata
            }

            yield {
                "_index": index_name,
                "_source": source
            }

    def bulk_index(self, index_name: str, raw_documents: List[Any], batch_size: int = 64):
        """배치 단위로 인덱싱 실행"""

        start_time = time.time()
        total_docs = len(raw_documents)

        for i in range(0, total_docs, batch_size):
            batch = raw_documents[i:i + batch_size]

            try:
                helpers.bulk(
                    self.es,
                    self._generate_actions(index_name, batch)
                )
            except Exception as e:
                print(f"[ERROR] Batch {i} failed: {e}")
                continue

            print(f"Progress: {min(i + batch_size, total_docs)}/{total_docs} indexed.")

        print(f"Indexing completed in {time.time() - start_time:.2f}s")

    def delete_indices_by_pattern(self, pattern: str, dry_run: bool = True):
        """
        패턴 기반 인덱스 삭제 (예: rag-* )

        Args:
            pattern (str): 삭제할 인덱스 패턴
            dry_run (bool): 실제 삭제하기전 실행될 내용 확인
        """
        indices = self.es.indices.get_alias(index=pattern)
        index_list = list(indices.keys())

        print(f"[INFO] Target indices: {index_list}")

        if dry_run:
            print("[DRY-RUN] No indices deleted.")
            return

        self.es.indices.delete(index=",".join(index_list))
        print(f"[SUCCESS] Deleted indices: {index_list}")

    def _flatten_metadata(self, metadata: dict) -> dict:
        """
        metadata 내부의 global_metadata를 1차원으로 펼침
        """
        if not metadata:
            return {}

        flat_meta = {}

        # 1. global_metadata 분리
        global_meta = metadata.get("global_metadata", {})

        # 2. 나머지 metadata
        local_meta = {
            k: v for k, v in metadata.items()
            if k != "global_metadata"
        }

        # 3. 병합 (충돌 시 local 우선)
        flat_meta.update(global_meta)
        flat_meta.update(local_meta)

        return flat_meta
    
    def _build_filter_clauses(self, filters: dict):
        """
        include / exclude 필터를 ES bool 구조로 변환
        """

        filter_clauses = []
        must_not_clauses = []

        if not filters:
            return filter_clauses, must_not_clauses

        # ✅ include (positive filter)
        include = filters.get("include", {})
        for k, v in include.items():
            if isinstance(v, list):
                filter_clauses.append({
                    "terms": {f"metadata.{k}": v}
                })
            else:
                filter_clauses.append({
                    "term": {f"metadata.{k}": v}
                })

        # ✅ exclude (negative filter)
        exclude = filters.get("exclude", {})
        for k, v in exclude.items():
            if isinstance(v, list):
                must_not_clauses.append({
                    "terms": {f"metadata.{k}": v}
                })
            else:
                must_not_clauses.append({
                    "term": {f"metadata.{k}": v}
                })

        return filter_clauses, must_not_clauses

    def hybrid_search(
        self,
        index_name: str,
        query: str,
        top_k: int = 10,
        num_candidates: int = 100,
        dense_weight: float = 0.5,
        sparse_weight: float = 0.2,
        bm25_weight: float = 0.2,
        filters: dict = None
        ):
        """
        BGE-M3 기반 Hybrid Search (Dense + Sparse + BM25 + Filter)

        Args:
            index_name (str)
            query (str)
            top_k (int): 최종 결과 개수
            num_candidates (int): knn 후보군
            dense_weight (float)
            sparse_weight (float)
            bm25_weight (float)
            filters (dict): metadata 필터

        Returns:
            List[dict]
        """

        # 1. query embedding 생성
        output = self.model.encode(
            [query],
            return_dense=True,
            return_sparse=True
        )

        query_dense = output["dense_vecs"][0].tolist()
        query_sparse = {
            str(k): float(v)
            for k, v in output["lexical_weights"][0].items()
            if v > 0
        }

        # 2. filter 구성
        filter_clauses = []

        if filters:
            for k, v in filters.items():

                # 리스트 → OR 조건
                if isinstance(v, list):
                    filter_clauses.append({
                        "terms": {f"metadata.{k}": v}
                    })

                # 단일 값
                else:
                    filter_clauses.append({
                        "term": {f"metadata.{k}": v}
                    })

        filter_clauses, must_not_clauses = self._build_filter_clauses(filters)

        # 3. hybrid query
        body = {
            "size": top_k,

            # 검색 응답 단계에서 벡터 제외
            "_source": {
                "excludes": ["dense_vector", "sparse_vector"]
                },

            # ✅ Dense (knn)
            "knn": {
                "field": "dense_vector",
                "query_vector": query_dense,
                "k": num_candidates,
                "num_candidates": num_candidates,
                "boost": dense_weight,
                **({"filter": filter_clauses} if filter_clauses else {})
            },

            # ✅ Sparse + BM25
            "query": {
                "bool": {
                    "should": [
                        # Sparse
                        {
                            "sparse_vector": {
                                "field": "sparse_vector",
                                "query_vector": query_sparse,
                                "boost": sparse_weight
                            }
                        },
                        # BM25
                        {
                            "match": {
                                "page_content": {
                                    "query": query,
                                    "boost": bm25_weight
                                }
                            }
                        }
                    ],
                    "filter": filter_clauses,      # positive filtering
                    "must_not": must_not_clauses   # negative filtering
                }
            }
        }

        response = self.es.search(index=index_name, body=body)

        # 4. 결과 정리
        results = []
        for hit in response["hits"]["hits"]:
            source = hit["_source"]

            EXCLUDE_FIELDS = {"page_content", "dense_vector", "sparse_vector", "metadata", "sparse_tokens"}
            meta = {
                k: v for k, v in source.items()
                if k not in EXCLUDE_FIELDS
            }

            # 기존 metadata merge
            if "metadata" in source:
                meta.update(source["metadata"])

            results.append({
                "score": hit["_score"],
                "page_content": source.get("page_content"),
                "metadata": self._flatten_metadata(meta)
            })

        return results
  
bem= BGEElasticManager()
bem

d:\auto_vectordb\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
bem.list_indices()

['.internal.alerts-transform.health.alerts-default-000001',
 '.internal.alerts-observability.logs.alerts-default-000001',
 '.internal.alerts-observability.uptime.alerts-default-000001',
 '.internal.alerts-ml.anomaly-detection.alerts-default-000001',
 '.internal.alerts-observability.slo.alerts-default-000001',
 'test-pos',
 '.internal.alerts-default.alerts-default-000001',
 '.internal.alerts-observability.apm.alerts-default-000001',
 '.internal.alerts-observability.metrics.alerts-default-000001',
 '.internal.alerts-ml.anomaly-detection-health.alerts-default-000001',
 '.internal.alerts-observability.threshold.alerts-default-000001',
 '.internal.alerts-security.alerts-default-000001',
 '.internal.alerts-stack.alerts-default-000001']

In [ ]:
bem.delete_indices_by_pattern(pattern="test-*", dry_run=True)

[INFO] Target indices: ['test-pos']
[SUCCESS] Deleted indices: ['test-pos']


In [5]:
index_name="test-pos"
bem.create_index(index_name=index_name)

[SUCCESS] Index created: test-pos


In [6]:
import pickle
with open("./docs/FWG_with_global.pkl", "rb") as f:
    loaded_text2 = pickle.load(f)
len(loaded_text2)

7

In [7]:
bem.bulk_index(index_name=index_name, raw_documents=loaded_text2)

You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Progress: 7/7 indexed.
Indexing completed in 35.38s


In [10]:
res = bem.hybrid_search(index_name=index_name, 
                        query="장비 주요 사양 요약", 
                        filters = {
                            "include": {
                                "ship_numbers": ["8250"]
                            },
                            "exclude": {
                                "ship_numbers": ["8300"]
                            }
                        }
                        )
res

[{'score': 0.37529078,
  'page_content': 'Global Metadata: {"title": "TECHNICAL SPECIFICATION FOR F.W. GENERATOR", "ship_numbers": ["8250", "8251"], "product_name": "F.W. GENERATOR", "specifications": "Low-pressure evaporating type (M/E jacket water heating), Shell & Tube, 25 ton/day capacity with 15% fouling margin, max 10 PPM salinity, M/E jacket cooling F.W. heating medium, S.W. cooling medium", "document_type": "Technical Specification"} \n\n Global Context: This chunk is the Package List section that immediately follows the document header and precedes the technical specifications, listing the F.W. generator unit and spare parts for ships 8250/8251. \n\n Content: <!-- 🖼️❌ Image not available. Please use `PdfPipelineOptions(generate_picture_images=True)` -->\n\n| PACKAGE LIST   | PACKAGE LIST   | PACKAGE LIST                           |      |                   |\n|----------------|----------------|----------------------------------------|------|-------------------|\n| POR NO.     

In [ ]:
res[0]